# Hybrid Quantum-Classical CVRP Solver (Unified)
## Yale Quantum Courier Challenge 2026
This notebook combines the entire multi-stage solver architecture into a single, standalone runable notebook.


## Step 0: Standard Library Imports


In [ ]:
import time
import math
import itertools
import copy
import json
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')



## Step 1: Common Utilities (Geometry & Distances)


In [ ]:
"""
common.py
=========
Core shared utilities: geometry helpers, distance matrix computation,
route cost / load / feasibility checks, and instance parsing.

All customer IDs are 1-indexed integers.
Depot is always index 0 in the distance matrix (and node list).
Routes are stored as lists of customer IDs (without depot at either end
in the internal representation).
"""

import math
import json
from pathlib import Path


# ---------------------------------------------------------------------------
# Distance helpers
# ---------------------------------------------------------------------------

def euclidean(a, b):
    """Euclidean distance between two (x, y) points."""
    return math.sqrt((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2)


def build_distance_matrix(nodes):
    """
    Build a symmetric distance matrix from a list of (x, y) tuples.

    Parameters
    ----------
    nodes : list of (float, float)
        Index 0 must be the depot; indices 1..n are customers.

    Returns
    -------
    D : list[list[float]]
        D[i][j] == D[j][i] == Euclidean distance between nodes[i] and nodes[j].
    """
    n = len(nodes)
    D = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(i + 1, n):
            d = euclidean(nodes[i], nodes[j])
            D[i][j] = d
            D[j][i] = d
    return D


# ---------------------------------------------------------------------------
# Route cost / load / angle helpers
# ---------------------------------------------------------------------------

def route_distance(route, D):
    """
    Total travel distance for a route (depot -> customers -> depot).

    Parameters
    ----------
    route : list[int]
        Customer IDs (1-indexed), no depot at ends.
    D : list[list[float]]
        Full distance matrix (depot = index 0).

    Returns
    -------
    float
    """
    if not route:
        return 0.0
    total = D[0][route[0]]
    for k in range(len(route) - 1):
        total += D[route[k]][route[k + 1]]
    total += D[route[-1]][0]
    return total


def route_load(route, demands):
    """
    Total demand on a route.

    Parameters
    ----------
    route : list[int]
        Customer IDs (1-indexed).
    demands : list[int]
        demands[i] is the demand of customer i (index 0 = depot, demand 0).

    Returns
    -------
    int
    """
    return sum(demands[c] for c in route)


def total_solution_distance(routes, D):
    """Sum of route_distance over all routes."""
    return sum(route_distance(r, D) for r in routes)


def customer_angle(depot, node):
    """
    Polar angle (radians) from depot to a customer node.

    Parameters
    ----------
    depot : (float, float)
    node  : (float, float)

    Returns
    -------
    float in [-pi, pi]
    """
    return math.atan2(node[1] - depot[1], node[0] - depot[0])


def route_centroid(route, nodes):
    """
    Centroid of customer positions in a route.

    Parameters
    ----------
    route : list[int]  -- customer IDs (1-indexed)
    nodes : list[(float, float)]  -- index 0 = depot

    Returns
    -------
    (float, float)
    """
    if not route:
        return nodes[0]
    xs = [nodes[c][0] for c in route]
    ys = [nodes[c][1] for c in route]
    return (sum(xs) / len(xs), sum(ys) / len(ys))


# ---------------------------------------------------------------------------
# Feasibility / validation
# ---------------------------------------------------------------------------

def is_route_feasible(route, demands, capacity):
    """
    Check that a route does not exceed capacity.

    Parameters
    ----------
    route    : list[int]
    demands  : list[int]
    capacity : int

    Returns
    -------
    bool
    """
    return route_load(route, demands) <= capacity


def validate_solution(routes, demands, capacity, num_customers, num_vehicles):
    """
    Full solution validity check.

    Rules
    -----
    - Every customer 1..num_customers appears exactly once across all routes.
    - No route exceeds capacity.
    - Number of routes <= num_vehicles.

    Returns
    -------
    bool
    """
    if len(routes) > num_vehicles:
        return False
    seen = []
    for route in routes:
        if route_load(route, demands) > capacity:
            return False
        seen.extend(route)
    return sorted(seen) == list(range(1, num_customers + 1))


# ---------------------------------------------------------------------------
# Instance parsing
# ---------------------------------------------------------------------------

def parse_instance_dict(instance):
    """
    Parse one instance dict (from the JSON benchmark file) into a canonical form.

    Parameters
    ----------
    instance : dict
        Keys: instance_id, Nv, C, customers (list of dicts with customer_id, x, y, demand).

    Returns
    -------
    dict with keys:
        instance_id : str
        num_vehicles : int
        capacity : int
        nodes : list[(float, float)]   -- index 0 = depot
        demands : list[int]            -- index 0 = 0 (depot), 1..n = customer demands
        num_customers : int
    """
    num_vehicles = int(instance["Nv"])
    capacity = int(instance["C"])
    instance_id = instance["instance_id"]

    depot = None
    raw_customers = []

    for c in instance["customers"]:
        cid = int(c["customer_id"])
        x = float(c["x"])
        y = float(c["y"])
        d = int(c["demand"])
        if cid == 0:
            depot = (x, y)
        else:
            raw_customers.append((cid, x, y, d))

    if depot is None:
        raise ValueError(f"Instance {instance_id} has no depot (customer_id=0).")

    # Sort by customer ID to ensure consistent ordering
    raw_customers.sort(key=lambda t: t[0])

    nodes = [depot] + [(c[1], c[2]) for c in raw_customers]
    demands = [0] + [c[3] for c in raw_customers]
    num_customers = len(raw_customers)

    return {
        "instance_id": instance_id,
        "num_vehicles": num_vehicles,
        "capacity": capacity,
        "nodes": nodes,
        "demands": demands,
        "num_customers": num_customers,
    }


def load_instances_json(json_path):
    """
    Load all instances from a benchmark JSON file.

    Parameters
    ----------
    json_path : str or Path

    Returns
    -------
    list[dict]  -- each dict is the output of parse_instance_dict
    """
    with open(json_path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    return [parse_instance_dict(inst) for inst in raw]


def challenge_instances():
    """
    Return the four small challenge instances in canonical form.
    These are defined inline -- no file dependency.

    Returns
    -------
    list[dict]
    """
    raw = [
        {
            "instance_id": "Instance1",
            "Nv": 2,
            "C": 5,
            "customers": [
                {"customer_id": 0, "x": 0, "y": 0, "demand": 0},
                {"customer_id": 1, "x": -2, "y": 2, "demand": 1},
                {"customer_id": 2, "x": -5, "y": 8, "demand": 1},
                {"customer_id": 3, "x":  2, "y": 3, "demand": 1},
            ],
        },
        {
            "instance_id": "Instance2",
            "Nv": 2,
            "C": 2,
            "customers": [
                {"customer_id": 0, "x": 0, "y": 0, "demand": 0},
                {"customer_id": 1, "x": -2, "y": 2, "demand": 1},
                {"customer_id": 2, "x": -5, "y": 8, "demand": 1},
                {"customer_id": 3, "x":  2, "y": 3, "demand": 1},
            ],
        },
        {
            "instance_id": "Instance3",
            "Nv": 3,
            "C": 2,
            "customers": [
                {"customer_id": 0, "x": 0, "y":  0, "demand": 0},
                {"customer_id": 1, "x": -2, "y":  2, "demand": 1},
                {"customer_id": 2, "x": -5, "y":  8, "demand": 1},
                {"customer_id": 3, "x":  2, "y":  3, "demand": 1},
                {"customer_id": 4, "x":  5, "y":  7, "demand": 1},
                {"customer_id": 5, "x":  2, "y":  4, "demand": 1},
                {"customer_id": 6, "x":  2, "y": -3, "demand": 1},
            ],
        },
        {
            "instance_id": "Instance4",
            "Nv": 4,
            "C": 3,
            "customers": [
                {"customer_id":  0, "x":  0, "y":  0, "demand": 0},
                {"customer_id":  1, "x": -2, "y":  2, "demand": 1},
                {"customer_id":  2, "x": -5, "y":  8, "demand": 1},
                {"customer_id":  3, "x":  6, "y":  3, "demand": 1},
                {"customer_id":  4, "x":  4, "y":  4, "demand": 1},
                {"customer_id":  5, "x":  3, "y":  2, "demand": 1},
                {"customer_id":  6, "x":  0, "y":  2, "demand": 1},
                {"customer_id":  7, "x": -2, "y":  3, "demand": 1},
                {"customer_id":  8, "x": -4, "y":  3, "demand": 1},
                {"customer_id":  9, "x":  2, "y":  3, "demand": 1},
                {"customer_id": 10, "x":  2, "y":  7, "demand": 1},
                {"customer_id": 11, "x": -2, "y":  5, "demand": 1},
                {"customer_id": 12, "x": -1, "y":  4, "demand": 1},
            ],
        },
    ]
    return [parse_instance_dict(inst) for inst in raw]


# ---------------------------------------------------------------------------
# Solution formatting helpers
# ---------------------------------------------------------------------------

def format_routes_text(routes, instance_id=""):
    """
    Format routes in the required hackathon submission format.

    Example output::

        r1: 0, 2, 3, 0
        r2: 0, 1, 0

    Parameters
    ----------
    routes : list[list[int]]  -- customer IDs, no depot
    instance_id : str

    Returns
    -------
    str
    """
    lines = []
    if instance_id:
        lines.append(f"# {instance_id}")
    for i, route in enumerate(routes, start=1):
        seq = [0] + route + [0]
        lines.append("r{}: {}".format(i, ", ".join(map(str, seq))))
    return "\n".join(lines)


def solution_summary(routes, D, demands, capacity, num_customers, num_vehicles, method=""):
    """
    Return a summary dict for a solved instance.

    Returns
    -------
    dict
    """
    dist = total_solution_distance(routes, D)
    valid = validate_solution(routes, demands, capacity, num_customers, num_vehicles)
    return {
        "method": method,
        "total_distance": round(dist, 6),
        "vehicles_used": len(routes),
        "vehicles_allowed": num_vehicles,
        "valid": valid,
        "routes": [
            {
                "vehicle": i + 1,
                "route": [0] + r + [0],
                "load": route_load(r, demands),
                "distance": round(route_distance(r, D), 6),
            }
            for i, r in enumerate(routes)
        ],
    }



## Step 2: Decomposition (Seed Generation)


In [ ]:
"""
decomposition.py
================
Seed portfolio: multiple classical decomposition strategies that each produce
a valid (or near-valid) initial route set for a CVRP instance.

Strategies implemented
----------------------
1. sweep               -- angular sweep from depot, standard direction
2. reverse_sweep       -- angular sweep, reverse direction
3. shifted_sweep       -- multiple angular offsets for the standard sweep
4. clarke_wright       -- savings-based heuristic
5. capacitated_kmeans  -- centroid-based clustering (no sklearn required)

Each strategy returns a CandidateSolution named-tuple (or dict).

Output contract
---------------
Every returned solution has these keys:
    routes       : list[list[int]]   customer IDs, no depot at ends
    total_dist   : float
    vehicles     : int               number of non-empty routes
    valid        : bool
    method       : str               short label
    meta         : dict              extra info (offset, direction, etc.)
"""

import math
import random
    build_distance_matrix,
    route_distance,
    total_solution_distance,
    validate_solution,
    customer_angle,
    route_load,
)


# ---------------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------------

def _make_result(routes, D, demands, capacity, num_customers, num_vehicles, method, meta=None):
    """Pack a route list into a standard result dict."""
    routes = [r for r in routes if r]  # drop empty routes
    dist = total_solution_distance(routes, D)
    valid = validate_solution(routes, demands, capacity, num_customers, num_vehicles)
    return {
        "routes": routes,
        "total_dist": round(dist, 6),
        "vehicles": len(routes),
        "valid": valid,
        "method": method,
        "meta": meta or {},
    }


def _split_into_routes(ordered_customers, demands, capacity, max_routes):
    """
    Pack customers (in given order) greedily into routes respecting capacity.
    Splits into at most max_routes routes; overflows go into extra routes.

    Returns list[list[int]]
    """
    routes = []
    current_route = []
    current_load = 0

    for c in ordered_customers:
        d = demands[c]
        if current_load + d > capacity and current_route:
            routes.append(current_route)
            current_route = []
            current_load = 0
        current_route.append(c)
        current_load += d

    if current_route:
        routes.append(current_route)

    return routes


# ---------------------------------------------------------------------------
# 1. Sweep decomposition
# ---------------------------------------------------------------------------

def sweep_decomposition(nodes, demands, capacity, num_vehicles, offset=0.0, reverse=False, method_name=None):
    """
    Angular sweep from depot -- sort customers by polar angle, then pack
    greedily into routes.

    Parameters
    ----------
    nodes       : list[(float, float)]  -- index 0 = depot
    demands     : list[int]
    capacity    : int
    num_vehicles: int
    offset      : float                 -- angle offset in radians
    reverse     : bool
    method_name : str or None

    Returns
    -------
    dict (standard result format)
    """
    if method_name is None:
        method_name = "reverse_sweep" if reverse else "sweep"

    depot = nodes[0]
    customers = list(range(1, len(nodes)))

    # Sort by polar angle (with offset, wrapped to [-pi, pi))
    def sort_key(c):
        angle = customer_angle(depot, nodes[c]) + offset
        # wrap to [-pi, pi)
        while angle > math.pi:
            angle -= 2 * math.pi
        while angle <= -math.pi:
            angle += 2 * math.pi
        return angle

    customers.sort(key=sort_key, reverse=reverse)

    D = build_distance_matrix(nodes)
    num_customers = len(nodes) - 1
    routes = _split_into_routes(customers, demands, capacity, num_vehicles)

    return _make_result(routes, D, demands, capacity, num_customers, num_vehicles, method_name,
                        meta={"offset": offset, "reverse": reverse})


def reverse_sweep(nodes, demands, capacity, num_vehicles):
    """Reverse angular sweep."""
    return sweep_decomposition(nodes, demands, capacity, num_vehicles,
                               offset=0.0, reverse=True, method_name="reverse_sweep")


def shifted_sweep_portfolio(nodes, demands, capacity, num_vehicles, n_offsets=4):
    """
    Try n_offsets evenly-spaced angular offsets and return all results.

    Returns
    -------
    list[dict]
    """
    results = []
    for k in range(n_offsets):
        offset = (2 * math.pi * k) / n_offsets
        r = sweep_decomposition(nodes, demands, capacity, num_vehicles,
                                offset=offset, method_name=f"sweep_off{k}")
        results.append(r)
    return results


# ---------------------------------------------------------------------------
# 2. Clarke-Wright savings
# ---------------------------------------------------------------------------

def clarke_wright(nodes, demands, capacity, num_vehicles):
    """
    Clarke-Wright savings heuristic.

    Returns
    -------
    dict (standard result format)
    """
    D = build_distance_matrix(nodes)
    num_customers = len(nodes) - 1
    customers = list(range(1, num_customers + 1))

    # Compute savings s(i,j) = d(0,i) + d(0,j) - d(i,j)
    savings = []
    for i in customers:
        for j in customers:
            if i >= j:
                continue
            s = D[0][i] + D[0][j] - D[i][j]
            savings.append((s, i, j))
    savings.sort(reverse=True)

    # Start: each customer on its own route
    routes = [[c] for c in customers]
    customer_to_route = {c: routes[idx] for idx, c in enumerate(customers)}

    def remap(route):
        for c in route:
            customer_to_route[c] = route

    for _, i, j in savings:
        ri = customer_to_route.get(i)
        rj = customer_to_route.get(j)

        if ri is None or rj is None or ri is rj:
            continue
        if route_load(ri, demands) + route_load(rj, demands) > capacity:
            continue
        if len(routes) <= num_vehicles and len(routes) - 1 == 0:
            # Only one route remaining, skip
            continue
        # Edge endpoints only
        if i not in (ri[0], ri[-1]) or j not in (rj[0], rj[-1]):
            continue

        left = ri[:] if ri[-1] == i else ri[::-1]
        right = rj[:] if rj[0] == j else rj[::-1]
        merged = left + right

        routes.remove(ri)
        routes.remove(rj)
        routes.append(merged)
        remap(merged)

    return _make_result(routes, D, demands, capacity, num_customers, num_vehicles,
                        "clarke_wright")


# ---------------------------------------------------------------------------
# 3. Capacitated k-means (no sklearn)
# ---------------------------------------------------------------------------

def _kmeans_assign(customers, centroids, nodes):
    """
    Assign each customer to closest centroid.
    Returns list[list[int]] of length k.
    """
    k = len(centroids)
    clusters = [[] for _ in range(k)]
    for c in customers:
        cx, cy = nodes[c]
        best = min(range(k), key=lambda ci: (cx - centroids[ci][0]) ** 2 + (cy - centroids[ci][1]) ** 2)
        clusters[best].append(c)
    return clusters


def _kmeans_centroids(clusters, nodes):
    """Recompute centroids from cluster assignments."""
    centroids = []
    for cluster in clusters:
        if not cluster:
            centroids.append((0.0, 0.0))
        else:
            xs = [nodes[c][0] for c in cluster]
            ys = [nodes[c][1] for c in cluster]
            centroids.append((sum(xs) / len(xs), sum(ys) / len(ys)))
    return centroids


def _balance_clusters(clusters, demands, capacity):
    """
    Post-k-means balancing: move overloaded cluster members to underloaded ones.
    Simple greedy reallocate -- not globally optimal, but practical.
    """
    changed = True
    while changed:
        changed = False
        for i, ci in enumerate(clusters):
            load_i = sum(demands[c] for c in ci)
            if load_i <= capacity:
                continue
            # Try to move customers to a cluster that can absorb them
            for j, cj in enumerate(clusters):
                if i == j:
                    continue
                load_j = sum(demands[c] for c in cj)
                # Try each customer in ci
                for c in list(ci):
                    d = demands[c]
                    if load_j + d <= capacity and load_i - d >= 0:
                        ci.remove(c)
                        cj.append(c)
                        load_i -= d
                        load_j += d
                        changed = True
                        break
                if load_i <= capacity:
                    break
    return clusters


def capacitated_kmeans(nodes, demands, capacity, num_vehicles, seed=42, max_iter=50):
    """
    Capacitated k-means: cluster customers into num_vehicles clusters,
    then build one route per cluster with a nearest-neighbor tour.

    Parameters
    ----------
    nodes       : list[(float, float)]
    demands     : list[int]
    capacity    : int
    num_vehicles: int
    seed        : int
    max_iter    : int

    Returns
    -------
    dict (standard result format)
    """
    D = build_distance_matrix(nodes)
    num_customers = len(nodes) - 1
    customers = list(range(1, num_customers + 1))
    k = min(num_vehicles, num_customers)

    rng = random.Random(seed)
    # Init centroids from a random sample of customer positions
    init_ids = rng.sample(customers, k)
    centroids = [(nodes[c][0], nodes[c][1]) for c in init_ids]

    clusters = [[] for _ in range(k)]
    for _ in range(max_iter):
        new_clusters = _kmeans_assign(customers, centroids, nodes)
        new_centroids = _kmeans_centroids(new_clusters, nodes)
        if new_centroids == centroids:
            clusters = new_clusters
            break
        centroids = new_centroids
        clusters = new_clusters

    # Balance overloaded clusters
    clusters = _balance_clusters(clusters, demands, capacity)

    # Build nearest-neighbor route within each cluster
    routes = []
    for cluster in clusters:
        if not cluster:
            continue
        if len(cluster) == 1:
            routes.append(cluster[:])
            continue
        # Nearest-neighbor from depot
        route = []
        remaining = set(cluster)
        current = 0
        while remaining:
            nxt = min(remaining, key=lambda c: D[current][c])
            route.append(nxt)
            remaining.remove(nxt)
            current = nxt
        routes.append(route)

    return _make_result(routes, D, demands, capacity, num_customers, num_vehicles,
                        "capacitated_kmeans", meta={"seed": seed})


# ---------------------------------------------------------------------------
# 4. Unified seed portfolio
# ---------------------------------------------------------------------------

def generate_seed_portfolio(nodes, demands, capacity, num_vehicles,
                            sweep_offsets=4, kmeans_seeds=None):
    """
    Generate all candidate decompositions and return them as a list.

    Parameters
    ----------
    nodes        : list[(float, float)]
    demands      : list[int]
    capacity     : int
    num_vehicles : int
    sweep_offsets: int    -- number of shifted sweep angles to try
    kmeans_seeds : list[int] or None

    Returns
    -------
    list[dict]  -- each is a standard result dict (may be invalid)
    """
    if kmeans_seeds is None:
        kmeans_seeds = [42, 7, 123]

    candidates = []

    # Sweep variants
    candidates.append(sweep_decomposition(nodes, demands, capacity, num_vehicles))
    candidates.append(reverse_sweep(nodes, demands, capacity, num_vehicles))
    candidates.extend(shifted_sweep_portfolio(nodes, demands, capacity, num_vehicles, sweep_offsets))

    # Clarke-Wright
    candidates.append(clarke_wright(nodes, demands, capacity, num_vehicles))

    # k-means variants
    for s in kmeans_seeds:
        candidates.append(capacitated_kmeans(nodes, demands, capacity, num_vehicles, seed=s))

    return candidates


def best_valid_candidate(candidates):
    """
    Return the valid candidate with the lowest total distance.
    Falls back to the overall best (valid or not) if none are valid.
    """
    valid = [c for c in candidates if c["valid"]]
    pool = valid if valid else candidates
    return min(pool, key=lambda c: c["total_dist"])



## Step 3: Local Search (2-Opt Cleanup)


In [ ]:
"""
local_search.py
===============
Classical intra-route local search improvements.

Implemented
-----------
- two_opt      : standard 2-opt improvement for a single route
- two_opt_all  : apply 2-opt to every route in a solution
- three_opt    : bounded 3-opt improvement (optional, slower)
- nearest_neighbor_route : greedy nearest-neighbor route builder
"""



# ---------------------------------------------------------------------------
# 2-opt
# ---------------------------------------------------------------------------

def two_opt(route, D, max_iter=None):
    """
    Improve a single route with 2-opt.  Only accepts improving moves.

    Parameters
    ----------
    route    : list[int]  -- customer IDs (no depot at ends)
    D        : list[list[float]]  -- distance matrix, depot = index 0
    max_iter : int or None  -- maximum passes; None = run until no improvement

    Returns
    -------
    list[int] -- improved route (new list, original unmodified)
    """
    best = route[:]
    improved = True
    iters = 0

    while improved:
        if max_iter is not None and iters >= max_iter:
            break
        improved = False
        iters += 1
        for i in range(len(best) - 1):
            for j in range(i + 2, len(best)):
                # Current cost of edge (i, i+1) + edge (j, j+1)
                # After reversal: edge (i, j) + edge (i+1, j+1)
                a = best[i]
                b = best[i + 1]
                c = best[j]
                d = best[j + 1] if j + 1 < len(best) else 0  # next node (0 = depot)

                prev_a = best[i - 1] if i > 0 else 0  # unused in standard 2-opt
                # Standard 2-opt: reverse the segment best[i+1 .. j]

                before = D[a][b] + D[c][d]
                after = D[a][c] + D[b][d]

                if after < before - 1e-10:
                    best[i + 1:j + 1] = best[i + 1:j + 1][::-1]
                    improved = True

    return best


def two_opt_all(routes, D, max_iter=None):
    """
    Apply 2-opt to every route in a solution.

    Parameters
    ----------
    routes   : list[list[int]]
    D        : list[list[float]]
    max_iter : int or None

    Returns
    -------
    list[list[int]]  -- new route list with improved routes
    """
    return [two_opt(r, D, max_iter=max_iter) for r in routes]


# ---------------------------------------------------------------------------
# 3-opt (bounded / simplified)
# ---------------------------------------------------------------------------

def three_opt(route, D, max_iter=3):
    """
    Simplified 3-opt: try all triple edge swaps that reduce route cost.
    Only the 2-opt-style reconnections are checked (not full 8-option 3-opt)
    to keep complexity manageable.  For routes with n > 20, this can be slow;
    caller should gate on route length.

    Parameters
    ----------
    route    : list[int]
    D        : list[list[float]]
    max_iter : int  -- maximum outer passes

    Returns
    -------
    list[int]
    """
    best = route[:]
    n = len(best)

    if n < 4:
        return best

    def segment_cost(r):
        return route_distance(r, D)

    for _ in range(max_iter):
        improved = False
        for i in range(n - 2):
            for j in range(i + 1, n - 1):
                for k in range(j + 1, n):
                    # Segment borders (depot = 0)
                    A = best[i]
                    B = best[i + 1]
                    C = best[j]
                    E = best[j + 1]
                    F = best[k]
                    G = best[k + 1] if k + 1 < n else 0

                    d0 = D[A][B] + D[C][E] + D[F][G]  # current

                    # Candidate reconnection: reverse segment j+1..k
                    d1 = D[A][B] + D[C][F] + D[E][G]
                    if d1 < d0 - 1e-10:
                        best[j + 1:k + 1] = best[j + 1:k + 1][::-1]
                        improved = True
                        continue

                    # Candidate reconnection: reverse segment i+1..j
                    d2 = D[A][C] + D[B][E] + D[F][G]
                    if d2 < d0 - 1e-10:
                        best[i + 1:j + 1] = best[i + 1:j + 1][::-1]
                        improved = True
                        continue

        if not improved:
            break

    return best


def three_opt_all(routes, D, max_route_len=20, max_iter=3):
    """
    Apply 3-opt to routes shorter than max_route_len; 2-opt for longer ones.

    Parameters
    ----------
    routes        : list[list[int]]
    D             : list[list[float]]
    max_route_len : int
    max_iter      : int

    Returns
    -------
    list[list[int]]
    """
    result = []
    for r in routes:
        if len(r) <= max_route_len:
            result.append(three_opt(r, D, max_iter=max_iter))
        else:
            result.append(two_opt(r, D))
    return result


# ---------------------------------------------------------------------------
# Nearest-neighbor route builder (standalone utility)
# ---------------------------------------------------------------------------

def nearest_neighbor_route(customers, D):
    """
    Build a single route visiting all given customers using a
    nearest-neighbor greedy heuristic starting from the depot (index 0).

    Parameters
    ----------
    customers : list[int]  -- customer IDs to visit
    D         : list[list[float]]

    Returns
    -------
    list[int]  -- ordered customer IDs
    """
    if not customers:
        return []
    remaining = list(customers)
    route = []
    current = 0  # depot
    while remaining:
        nxt = min(remaining, key=lambda c: D[current][c])
        route.append(nxt)
        remaining.remove(nxt)
        current = nxt
    return route


# ---------------------------------------------------------------------------
# Cleanup pass
# ---------------------------------------------------------------------------

def cleanup_solution(routes, D, use_three_opt=False, three_opt_threshold=15):
    """
    Apply 2-opt (and optionally 3-opt) to all routes.

    Parameters
    ----------
    routes              : list[list[int]]
    D                   : list[list[float]]
    use_three_opt       : bool
    three_opt_threshold : int  -- apply 3-opt only if route len <= this

    Returns
    -------
    list[list[int]]
    """
    if use_three_opt:
        return three_opt_all(routes, D, max_route_len=three_opt_threshold)
    return two_opt_all(routes, D)



## Step 4: Inter-Route Repair (Relocate & Swap)


In [ ]:
"""
repair.py
=========
Inter-route local search: relocate and swap moves.

Both moves respect capacity and only accept improving moves (strictly better
total distance).  After each accepted move, 2-opt is re-applied to the two
touched routes only.

Design notes
------------
- Works on general route sets, not just sweep-style sectors.
- Route pairs are ordered by centroid proximity to try the most promising
  pairs first.
- This module has no quantum dependency.
"""

import math
    route_distance,
    route_load,
    total_solution_distance,
    route_centroid,
)


# ---------------------------------------------------------------------------
# Proximity ordering
# ---------------------------------------------------------------------------

def _route_pair_order(routes, nodes):
    """
    Return all distinct route index pairs (i, j) sorted by centroid distance
    (closest centroids first -- most likely to benefit from inter-route moves).

    Parameters
    ----------
    routes : list[list[int]]
    nodes  : list[(float, float)]

    Returns
    -------
    list[(int, int)]
    """
    centroids = [route_centroid(r, nodes) for r in routes]
    pairs = []
    for i in range(len(routes)):
        for j in range(i + 1, len(routes)):
            cx = centroids[i][0] - centroids[j][0]
            cy = centroids[i][1] - centroids[j][1]
            dist = math.sqrt(cx * cx + cy * cy)
            pairs.append((dist, i, j))
    pairs.sort()
    return [(i, j) for (_, i, j) in pairs]


# ---------------------------------------------------------------------------
# Relocate move: move one customer from route_a to route_b
# ---------------------------------------------------------------------------

def _try_relocate(routes, D, demands, capacity, nodes, max_pair_fraction=1.0):
    """
    Try all one-customer relocations.  Accept first improving move (first-
    improvement strategy).

    Parameters
    ----------
    routes             : list[list[int]]  -- modified in place on improvement
    D                  : list[list[float]]
    demands            : list[int]
    capacity           : int
    nodes              : list[(float, float)]
    max_pair_fraction  : float in (0,1]  -- fraction of route pairs to check

    Returns
    -------
    bool  -- True if an improvement was found
    """
    pairs = _route_pair_order(routes, nodes)
    n_check = max(1, int(len(pairs) * max_pair_fraction))
    pairs = pairs[:n_check]

    for i, j in pairs:
        ri = routes[i]
        rj = routes[j]

        # Try moving each customer from ri -> rj
        for direction in [(i, j), (j, i)]:
            src_idx, dst_idx = direction
            src = routes[src_idx]
            dst = routes[dst_idx]

            if not src:
                continue

            d_src_before = route_distance(src, D)
            d_dst_before = route_distance(dst, D)
            base = d_src_before + d_dst_before

            for pos, cust in enumerate(src):
                if route_load(dst, demands) + demands[cust] > capacity:
                    continue

                # Remove from src
                new_src = src[:pos] + src[pos + 1:]

                # Find best insertion point in dst
                best_gain = 1e-10  # must be strictly positive
                best_insert = -1
                d_new_src = route_distance(new_src, D) if new_src else 0.0

                for ins in range(len(dst) + 1):
                    new_dst = dst[:ins] + [cust] + dst[ins:]
                    d_new_dst = route_distance(new_dst, D)
                    gain = base - (d_new_src + d_new_dst)
                    if gain > best_gain:
                        best_gain = gain
                        best_insert = ins

                if best_insert >= 0:
                    # Accept move
                    new_src_opt = two_opt(new_src, D) if new_src else []
                    new_dst_raw = dst[:best_insert] + [cust] + dst[best_insert:]
                    new_dst_opt = two_opt(new_dst_raw, D)

                    routes[src_idx] = new_src_opt
                    routes[dst_idx] = new_dst_opt
                    # Remove empty routes
                    routes[:] = [r for r in routes if r]
                    return True

    return False


# ---------------------------------------------------------------------------
# Swap move: exchange one customer between route_a and route_b
# ---------------------------------------------------------------------------

def _try_swap(routes, D, demands, capacity, nodes, max_pair_fraction=1.0):
    """
    Try all one-for-one customer swaps between route pairs.
    Accept first improving move.

    Returns
    -------
    bool
    """
    pairs = _route_pair_order(routes, nodes)
    n_check = max(1, int(len(pairs) * max_pair_fraction))
    pairs = pairs[:n_check]

    for i, j in pairs:
        ri = routes[i]
        rj = routes[j]

        if not ri or not rj:
            continue

        d_ri = route_distance(ri, D)
        d_rj = route_distance(rj, D)
        base = d_ri + d_rj

        for ci_pos, ci in enumerate(ri):
            for cj_pos, cj in enumerate(rj):
                # Capacity check after swap
                new_load_i = route_load(ri, demands) - demands[ci] + demands[cj]
                new_load_j = route_load(rj, demands) - demands[cj] + demands[ci]
                if new_load_i > capacity or new_load_j > capacity:
                    continue

                new_ri = ri[:ci_pos] + [cj] + ri[ci_pos + 1:]
                new_rj = rj[:cj_pos] + [ci] + rj[cj_pos + 1:]

                d_new_ri = route_distance(new_ri, D)
                d_new_rj = route_distance(new_rj, D)

                if base - (d_new_ri + d_new_rj) > 1e-10:
                    # Accept: apply 2-opt and update
                    routes[i] = two_opt(new_ri, D)
                    routes[j] = two_opt(new_rj, D)
                    return True

    return False


# ---------------------------------------------------------------------------
# Public repair interface
# ---------------------------------------------------------------------------

def repair_routes(routes, D, demands, capacity, nodes,
                  max_iter=50, use_relocate=True, use_swap=True,
                  max_pair_fraction=1.0):
    """
    Iteratively apply relocate and swap moves until no improvement is found
    or max_iter is reached.

    Parameters
    ----------
    routes            : list[list[int]]  -- modified in place
    D                 : list[list[float]]
    demands           : list[int]
    capacity          : int
    nodes             : list[(float, float)]
    max_iter          : int
    use_relocate      : bool
    use_swap          : bool
    max_pair_fraction : float  -- limit route pairs checked (1.0 = all)

    Returns
    -------
    list[list[int]]   -- improved routes (same object, also returned for clarity)
    float             -- distance gain (>= 0)
    """
    before = total_solution_distance(routes, D)

    for _ in range(max_iter):
        improved = False

        if use_relocate:
            if _try_relocate(routes, D, demands, capacity, nodes, max_pair_fraction):
                improved = True
                continue  # restart with relocate

        if use_swap:
            if _try_swap(routes, D, demands, capacity, nodes, max_pair_fraction):
                improved = True
                continue

        if not improved:
            break

    after = total_solution_distance(routes, D)
    gain = before - after
    return routes, max(0.0, round(gain, 6))



## Step 5: Quantum Local Improvement (QAOA)


In [ ]:
"""
quantum_improve.py
==================
Local neighborhood QAOA improvement for individual routes.

Architecture
------------
For each route we:
1. Select a small contiguous neighborhood of k customers.
2. Keep left-anchor and right-anchor nodes fixed (may be depot = 0).
3. Build an ANCHORED position-QUBO on those k customers.
4. Run QAOA (or brute-force for tiny k) and decode the best valid result.
5. Splice improved ordering back into the route only if it reduces cost.



QUBO design (anchored)
-----------------------
- Variables x[i, s] = 1 if the i-th neighborhood customer is at position s.
- n = k customers -> k^2 binary variables.
- Anchor costs:
    * from left_anchor to customer at position 0
    * from customer at position k-1 to right_anchor
- Standard one-hot row / column constraints.
- Objective = total intra-segment + anchor connection distance.

Scaling rule
------------
- max_local_qaoa_qubits (default 25) controls k.
- k = floor(sqrt(max_local_qaoa_qubits)).

Simulator backend
-----------------
- Uses qiskit_aer's AerSimulator (or Aer's StatevectorSimulator) for
  fast statevector simulation when QAOA is triggered.
- Falls back to qiskit.primitives.StatevectorEstimator if Aer unavailable.

Dependencies
------------
- qiskit
- qiskit_aer (recommended for speed)
- scipy.optimize.minimize
- numpy
"""

import time
import math
import itertools
import numpy as np


# ---------------------------------------------------------------------------
# Lazy imports: Qiskit and Aer
# ---------------------------------------------------------------------------

_qiskit_available = False
_aer_available = False

try:
    from qiskit.circuit.library import QAOAAnsatz
    from qiskit.quantum_info import SparsePauliOp
    _qiskit_available = True
except ImportError:
    pass

if _qiskit_available:
    try:
        from qiskit_aer import AerSimulator
        from qiskit_aer.primitives import EstimatorV2 as AerEstimatorV2
        from qiskit_aer.primitives import SamplerV2 as AerSamplerV2
        _aer_available = True
    except ImportError:
        pass

    if not _aer_available:
        try:
            from qiskit.primitives import StatevectorEstimator, StatevectorSampler
        except ImportError:
            _qiskit_available = False

    try:
        from scipy.optimize import minimize as scipy_minimize
    except ImportError:
        _qiskit_available = False


# ---------------------------------------------------------------------------
# QUBO / Ising helpers (anchored version)
# ---------------------------------------------------------------------------

def _build_anchored_qubo(neighborhood, D, left_anchor, right_anchor):
    """
    Build upper-triangular QUBO for ordering `neighborhood` customers given
    fixed left and right anchor nodes.

    Variables: x[i, s] -- customer i at position s (both 0-indexed).
    n = len(neighborhood) -> n^2 variables; qubit k = i*n + s.
    """
    n = len(neighborhood)
    N = n * n
    Q = np.zeros((N, N))

    def idx(i, s):
        return i * n + s

    dists = [D[a][b]
             for a in [left_anchor] + list(neighborhood)
             for b in [left_anchor, right_anchor] + list(neighborhood)
             if a != b]
    A = (max(dists) if dists else 1.0) * n * 2.0

    # Row constraint: each customer in exactly one position
    for i in range(n):
        for s in range(n):
            Q[idx(i, s), idx(i, s)] -= A
        for s in range(n):
            for t in range(s + 1, n):
                Q[idx(i, s), idx(i, t)] += 2 * A

    # Column constraint: each position holds exactly one customer
    for s in range(n):
        for i in range(n):
            Q[idx(i, s), idx(i, s)] -= A
        for i in range(n):
            for j in range(i + 1, n):
                Q[idx(i, s), idx(j, s)] += 2 * A

    # Objective: anchor -> pos 0
    for i, ci in enumerate(neighborhood):
        Q[idx(i, 0), idx(i, 0)] += D[left_anchor][ci]

    # Objective: pos n-1 -> right_anchor
    for i, ci in enumerate(neighborhood):
        Q[idx(i, n - 1), idx(i, n - 1)] += D[ci][right_anchor]

    # Objective: intra-segment transitions
    for i, ci in enumerate(neighborhood):
        for j, cj in enumerate(neighborhood):
            if i == j:
                continue
            for s in range(n - 1):
                a_idx = idx(i, s)
                b_idx = idx(j, s + 1)
                lo, hi = min(a_idx, b_idx), max(a_idx, b_idx)
                Q[lo, hi] += D[ci][cj]

    return Q


def _qubo_to_ising(Q):
    """Convert upper-triangular QUBO to Ising SparsePauliOp."""
    n = Q.shape[0]
    constant = 0.0
    h = np.zeros(n)
    J = {}

    for k in range(n):
        constant += Q[k, k] / 2.0
        h[k] -= Q[k, k] / 2.0

    for k in range(n):
        for l in range(k + 1, n):
            if abs(Q[k, l]) > 1e-12:
                constant += Q[k, l] / 4.0
                h[k] -= Q[k, l] / 4.0
                h[l] -= Q[k, l] / 4.0
                J[(k, l)] = Q[k, l] / 4.0

    pauli_list = []
    for k in range(n):
        if abs(h[k]) > 1e-12:
            lbl = ["I"] * n
            lbl[n - 1 - k] = "Z"
            pauli_list.append(("".join(lbl), h[k]))

    for (k, l), coef in J.items():
        if abs(coef) > 1e-12:
            lbl = ["I"] * n
            lbl[n - 1 - k] = "Z"
            lbl[n - 1 - l] = "Z"
            pauli_list.append(("".join(lbl), coef))

    if not pauli_list:
        pauli_list = [("I" * n, 0.0)]

    return SparsePauliOp.from_list(pauli_list).simplify(), constant


def _decode_bitstring(bitstring, n, neighborhood, D, left_anchor, right_anchor):
    """
    Decode a QAOA measurement bitstring into an ordered customer list.
    Returns (ordered_customers, segment_cost) or (None, inf) on violation.
    """
    bits = [int(b) for b in reversed(bitstring)]
    while len(bits) < n * n:
        bits.append(0)

    x = np.zeros((n, n), dtype=int)
    for i in range(n):
        for s in range(n):
            x[i, s] = bits[i * n + s]

    if not all(x[i, :].sum() == 1 for i in range(n)):
        return None, float("inf")
    if not all(x[:, s].sum() == 1 for s in range(n)):
        return None, float("inf")

    order = [None] * n
    for i in range(n):
        s = int(np.argmax(x[i, :]))
        order[s] = neighborhood[i]

    cost = D[left_anchor][order[0]]
    for k in range(n - 1):
        cost += D[order[k]][order[k + 1]]
    cost += D[order[-1]][right_anchor]

    return order, cost


# ---------------------------------------------------------------------------
# Neighborhood selection
# ---------------------------------------------------------------------------

def _worst_edges_neighborhood(route, D, k):
    """
    Select a contiguous window of k customers centred on the worst-cost
    contiguous segment.

    Returns (start_idx, end_idx) as slice into `route`.
    """
    n = len(route)
    if n <= k:
        return 0, n

    full = [0] + route + [0]
    # Sum of k consecutive interior edge costs as sliding window
    best_start = 0
    best_cost = -1.0
    for start in range(n - k + 1):
        # window covers full[start+1 .. start+k+1], i.e., k+1 edges
        win_cost = sum(D[full[start + t]][full[start + t + 1]] for t in range(k + 1))
        if win_cost > best_cost:
            best_cost = win_cost
            best_start = start

    end = best_start + k
    return best_start, end


def _random_neighborhood(route, k, rng):
    """Return a random contiguous neighborhood of exactly k customers."""
    n = len(route)
    if n <= k:
        return 0, n
    start = rng.randint(0, n - k)
    return start, start + k





# ---------------------------------------------------------------------------
# QAOA optimizer (only called for k >= 7 with Aer)
# ---------------------------------------------------------------------------

def _get_estimator_and_sampler():
    """Return (estimator, sampler) using Aer if available, else statevector."""
    if _aer_available:
        estimator = AerEstimatorV2()
        sampler = AerSamplerV2()
        return estimator, sampler
    else:
        from qiskit.primitives import StatevectorEstimator, StatevectorSampler  # noqa
        return StatevectorEstimator(), StatevectorSampler()


def _qaoa_optimize_neighborhood(neighborhood, D, left_anchor, right_anchor,
                                 reps=2, shots=2048, restarts=2,
                                 cobyla_maxiter=150, seed=None):
    """
    Run QAOA on the ordering of `neighborhood` customers with fixed anchors.
    Only called when len(neighborhood) >= 7 (otherwise brute force is used).

    Returns
    -------
    (best_order, best_cost, n_qubits, n_gates, elapsed_s)
    """
    n = len(neighborhood)
    t0 = time.time()

    Q = _build_anchored_qubo(neighborhood, D, left_anchor, right_anchor)
    H, _offset = _qubo_to_ising(Q)

    ansatz = QAOAAnsatz(H, reps=reps)
    from qiskit import transpile
    if _aer_available:
        from qiskit_aer import AerSimulator
        ansatz = transpile(ansatz, backend=AerSimulator(), optimization_level=1)
    else:
        ansatz = transpile(ansatz, basis_gates=['rx', 'ry', 'rz', 'cx', 'u1', 'u2', 'u3', 'p', 'u'], optimization_level=1)
    estimator, sampler = _get_estimator_and_sampler()

    rng = np.random.default_rng(seed)

    # Cost function wraps estimator call
    if _aer_available:
        def cost_fn(params):
            job = estimator.run([(ansatz, H, params)])
            result = job.result()
            return float(result[0].data.evs)
    else:
        def cost_fn(params):
            result = estimator.run([(ansatz, H, params)]).result()
            return float(result[0].data.evs)

    best_params = None
    best_energy = float("inf")
    for _ in range(restarts):
        x0 = rng.uniform(-np.pi, np.pi, ansatz.num_parameters)
        res = scipy_minimize(cost_fn, x0, method="COBYLA",
                             options={"maxiter": cobyla_maxiter})
        if res.fun < best_energy:
            best_energy = res.fun
            best_params = res.x

    # Sample the optimized circuit
    ansatz_meas = ansatz.measure_all(inplace=False)
    bound = ansatz_meas.assign_parameters(best_params)

    if _aer_available:
        counts = sampler.run([bound], shots=shots).result()[0].data.meas.get_counts()
    else:
        counts = sampler.run([bound], shots=shots).result()[0].data.meas.get_counts()

    best_order = None
    best_cost = float("inf")
    for bs in counts:
        order, cost = _decode_bitstring(bs, n, neighborhood, D, left_anchor, right_anchor)
        if order is not None and cost < best_cost:
            best_cost = cost
            best_order = order



    elapsed = time.time() - t0
    n_qubits = ansatz.num_qubits
    n_gates = sum(ansatz.decompose().count_ops().values())

    return best_order, best_cost, n_qubits, n_gates, elapsed


# ---------------------------------------------------------------------------
# Public interface: improve one route
# ---------------------------------------------------------------------------

def improve_route_qaoa(route, D,
                       max_local_qaoa_qubits=25,
                       strategy="worst_edges",
                       reps=2,
                       shots=2048,
                       restarts=2,
                       cobyla_maxiter=150,
                       seed=None,
                       rng=None):
    """
    Attempt to improve a single route using anchored local optimization.

    This purely uses QAOA simulation for any neighborhood size up to the qubit budget.

    Parameters
    ----------
    route                  : list[int]  -- customer IDs, no depot
    D                      : list[list[float]]
    max_local_qaoa_qubits  : int        -- qubit budget; k = floor(sqrt(budget))
    strategy               : str        -- "worst_edges" or "random"
    reps                   : int        -- QAOA layers
    shots                  : int
    restarts               : int        -- random restarts for COBYLA
    cobyla_maxiter         : int        -- COBYLA iterations per restart
    seed                   : int or None
    rng                    : random.Random instance (for "random" strategy)
    brute_force_threshold  : int        -- k <= this uses brute force (default 6)

    Returns
    -------
    improved_route  : list[int]
    gain            : float   -- distance reduction (>= 0)
    meta            : dict
    """
    if not _qiskit_available and not True:  # classical path always available
        pass

    if len(route) < 2:
        return route[:], 0.0, {"skipped": "route_too_short"}

    k = max(2, int(math.isqrt(max_local_qaoa_qubits)))
    k = min(k, len(route))

    if strategy == "worst_edges":
        start, end = _worst_edges_neighborhood(route, D, k)
    else:
        import random as _random
        _rng = rng if rng is not None else _random.Random(seed)
        start, end = _random_neighborhood(route, k, _rng)

    neighborhood = route[start:end]
    left_anchor = route[start - 1] if start > 0 else 0
    right_anchor = route[end] if end < len(route) else 0

    # Current cost of the segment (including anchor connections)
    seg_cost_before = D[left_anchor][neighborhood[0]]
    for i in range(len(neighborhood) - 1):
        seg_cost_before += D[neighborhood[i]][neighborhood[i + 1]]
    seg_cost_before += D[neighborhood[-1]][right_anchor]

    if not _qiskit_available:
        return route[:], 0.0, {"skipped": "qiskit_not_available"}
    best_order, best_cost, n_qubits, n_gates, elapsed = _qaoa_optimize_neighborhood(
        neighborhood, D, left_anchor, right_anchor,
        reps=reps, shots=shots, restarts=restarts,
        cobyla_maxiter=cobyla_maxiter, seed=seed,
    )
    meta = {
        "n_qubits": n_qubits,
        "n_gates": n_gates,
        "elapsed_s": round(elapsed, 3),
        "neighborhood_size": k,
        "strategy": strategy,
        "qaoa_reps": reps,
        "method": "qaoa",
    }

    if best_order is None:
        return route[:], 0.0, {**meta, "outcome": "no_valid_result"}

    gain = seg_cost_before - best_cost
    if gain <= 1e-10:
        return route[:], 0.0, {**meta, "outcome": "no_improvement"}

    new_route = route[:start] + best_order + route[end:]
    meta["outcome"] = "improved"
    return new_route, round(gain, 6), meta


# ---------------------------------------------------------------------------
# Apply to all routes
# ---------------------------------------------------------------------------

def improve_all_routes_qaoa(routes, D,
                             max_local_qaoa_qubits=25,
                             strategy="worst_edges",
                             reps=2,
                             shots=2048,
                             restarts=2,
                             cobyla_maxiter=150,
                             seed=None):
    """
    Apply improve_route_qaoa to every route.

    Returns
    -------
    improved_routes : list[list[int]]
    total_gain      : float
    all_meta        : list[dict]
    """
    improved_routes = []
    total_gain = 0.0
    all_meta = []

    for i, r in enumerate(routes):
        r_seed = seed + i if seed is not None else None
        new_r, gain, meta = improve_route_qaoa(
            r, D,
            max_local_qaoa_qubits=max_local_qaoa_qubits,
            strategy=strategy,
            reps=reps,
            shots=shots,
            restarts=restarts,
            cobyla_maxiter=cobyla_maxiter,
            seed=r_seed,
        )
        improved_routes.append(new_r)
        total_gain += gain
        all_meta.append(meta)

    return improved_routes, round(total_gain, 6), all_meta


# ---------------------------------------------------------------------------
# Standalone QAOA demo for one neighborhood (notebook helper)
# ---------------------------------------------------------------------------

def demo_qaoa_neighborhood(neighborhood, D, left_anchor, right_anchor,
                            reps=2, shots=2048, restarts=2, seed=42):
    """
    Run QAOA on a single neighborhood and return full diagnostics.
    Intended for notebook demonstration of the quantum subroutine.

    Returns
    -------
    dict with keys: order, cost, n_qubits, n_gates, elapsed_s, qubo_size
    """
    n = len(neighborhood)
    t0 = time.time()
    best_order, best_cost, n_qubits, n_gates, elapsed = _qaoa_optimize_neighborhood(
        neighborhood, D, left_anchor, right_anchor,
        reps=reps, shots=shots, restarts=restarts, seed=seed,
    )
    return {
        "order": best_order,
        "cost": best_cost,
        "n_qubits": n_qubits,
        "n_gates": n_gates,
        "elapsed_s": elapsed,
        "qubo_size": n * n,
    }



## Step 6: Hybrid Solver Orchestration


In [ ]:
"""
hybrid_solver.py
================
Top-level orchestration for the hybrid CVRP solver.

Pipeline stages
---------------
A. Seed generation    -- seed portfolio (sweep, CW, k-means)
B. Classical cleanup  -- 2-opt on every route in every candidate
C. Boundary repair    -- inter-route relocate + swap
D. Quantum local      -- anchored QAOA on bounded neighborhoods
E. Reporting          -- metrics for each stage

Usage
-----
    from hybrid_solver import HybridSolver

    solver = HybridSolver(inst, max_local_qaoa_qubits=25)
    result = solver.solve(run_quantum=True)
    print(result)
"""

import time
import copy

    build_distance_matrix,
    total_solution_distance,
    validate_solution,
    solution_summary,
)


class HybridSolver:
    """
    Orchestrates the full hybrid CVRP pipeline for one instance.

    Parameters
    ----------
    inst : dict
        Canonical instance dict from common.parse_instance_dict().
        Keys: instance_id, num_vehicles, capacity, nodes, demands, num_customers.

    max_local_qaoa_qubits : int
        Qubit budget for QAOA local improvement.  k = floor(sqrt(budget)).

    sweep_offsets : int
        Number of angular offsets for shifted sweep.

    kmeans_seeds : list[int] or None
        Seeds for k-means decomposition.

    seed : int
        Master random seed for reproducibility.

    use_three_opt : bool
        Apply 3-opt (slower) in addition to 2-opt.

    repair_max_iter : int
        Maximum inter-route repair iterations.

    qaoa_reps : int
        QAOA circuit depth (layers).

    qaoa_shots : int
        Measurement shots per QAOA run.

    qaoa_restarts : int
        COBYLA random restarts per neighborhood.

    verbose : bool
        Print progress.
    """

    def __init__(self,
                 inst,
                 max_local_qaoa_qubits=25,
                 sweep_offsets=4,
                 kmeans_seeds=None,
                 seed=42,
                 use_three_opt=False,
                 repair_max_iter=50,
                 qaoa_reps=2,
                 qaoa_shots=2048,
                 qaoa_restarts=2,
                 qaoa_cobyla_maxiter=150,
                 verbose=True):

        self.inst = inst
        self.max_local_qaoa_qubits = max_local_qaoa_qubits
        self.sweep_offsets = sweep_offsets
        self.kmeans_seeds = kmeans_seeds or [42, 7, 123]
        self.seed = seed
        self.use_three_opt = use_three_opt
        self.repair_max_iter = repair_max_iter
        self.qaoa_reps = qaoa_reps
        self.qaoa_shots = qaoa_shots
        self.qaoa_restarts = qaoa_restarts
        self.qaoa_cobyla_maxiter = qaoa_cobyla_maxiter
        self.verbose = verbose

        self.nodes = inst["nodes"]
        self.demands = inst["demands"]
        self.capacity = inst["capacity"]
        self.num_vehicles = inst["num_vehicles"]
        self.num_customers = inst["num_customers"]
        self.instance_id = inst["instance_id"]

        self.D = build_distance_matrix(self.nodes)

    def _log(self, msg):
        if self.verbose:
            print(f"[{self.instance_id}] {msg}")

    # ------------------------------------------------------------------
    # Stage A: seed portfolio
    # ------------------------------------------------------------------

    def stage_a_seeds(self):
        """Generate all decomposition candidates."""
        t0 = time.perf_counter()
        candidates = generate_seed_portfolio(
            self.nodes, self.demands, self.capacity, self.num_vehicles,
            sweep_offsets=self.sweep_offsets,
            kmeans_seeds=self.kmeans_seeds,
        )
        rt = time.perf_counter() - t0
        valid = [c for c in candidates if c["valid"]]
        best = best_valid_candidate(candidates)
        self._log(f"Stage A: {len(candidates)} seeds, {len(valid)} valid, "
                  f"best={best['method']} dist={best['total_dist']:.4f} ({rt:.3f}s)")
        return candidates, best

    # ------------------------------------------------------------------
    # Stage B: classical cleanup (2-opt)
    # ------------------------------------------------------------------

    def stage_b_cleanup(self, candidates):
        """Apply 2-opt to all routes in every candidate."""
        t0 = time.perf_counter()
        cleaned = []
        for cand in candidates:
            routes = cleanup_solution(
                cand["routes"], self.D,
                use_three_opt=self.use_three_opt
            )
            d = total_solution_distance(routes, self.D)
            valid = validate_solution(routes, self.demands, self.capacity,
                                      self.num_customers, self.num_vehicles)
            cleaned.append({**cand, "routes": routes, "total_dist": round(d, 6), "valid": valid})
        best = best_valid_candidate(cleaned)
        rt = time.perf_counter() - t0
        self._log(f"Stage B: best after cleanup={best['method']} dist={best['total_dist']:.4f} ({rt:.3f}s)")
        return cleaned, best

    # ------------------------------------------------------------------
    # Stage C: inter-route repair
    # ------------------------------------------------------------------

    def stage_c_repair(self, best_seed):
        """Apply relocate/swap repair on the best seed candidate."""
        t0 = time.perf_counter()
        routes = copy.deepcopy(best_seed["routes"])
        before = total_solution_distance(routes, self.D)

        routes, gain = repair_routes(
            routes, self.D, self.demands, self.capacity, self.nodes,
            max_iter=self.repair_max_iter,
        )

        # Post-repair 2-opt
        routes = cleanup_solution(routes, self.D, use_three_opt=self.use_three_opt)

        after = total_solution_distance(routes, self.D)
        valid = validate_solution(routes, self.demands, self.capacity,
                                  self.num_customers, self.num_vehicles)
        rt = time.perf_counter() - t0
        self._log(f"Stage C: repair gain={before - after:.4f}, "
                  f"dist={after:.4f}, valid={valid} ({rt:.3f}s)")
        return routes, round(before - after, 6)

    # ------------------------------------------------------------------
    # Stage D: local QAOA improvement
    # ------------------------------------------------------------------

    def stage_d_qaoa(self, routes):
        """Apply bounded local QAOA improvement to all routes."""
        t0 = time.perf_counter()
        before = total_solution_distance(routes, self.D)

        improved, gain, meta = improve_all_routes_qaoa(
            routes, self.D,
            max_local_qaoa_qubits=self.max_local_qaoa_qubits,
            strategy="worst_edges",
            reps=self.qaoa_reps,
            shots=self.qaoa_shots,
            restarts=self.qaoa_restarts,
            cobyla_maxiter=self.qaoa_cobyla_maxiter,
            seed=self.seed,
        )

        # Post-QAOA 2-opt
        improved = cleanup_solution(improved, self.D)

        after = total_solution_distance(improved, self.D)
        valid = validate_solution(improved, self.demands, self.capacity,
                                  self.num_customers, self.num_vehicles)
        rt = time.perf_counter() - t0

        qubit_estimates = [m.get("n_qubits", 0) for m in meta]
        self._log(f"Stage D: QAOA gain={before - after:.4f}, dist={after:.4f}, "
                  f"valid={valid}, max_qubits={max(qubit_estimates, default=0)} ({rt:.3f}s)")
        return improved, round(before - after, 6), meta

    # ------------------------------------------------------------------
    # solve()  -- full pipeline
    # ------------------------------------------------------------------

    def solve(self, run_quantum=True):
        """
        Run the full hybrid pipeline and return a comprehensive result dict.

        Parameters
        ----------
        run_quantum : bool
            If False, skip Stage D (useful for classical-only benchmarking).

        Returns
        -------
        dict with keys:
            instance_id
            stages dict with per-stage metrics
            routes_final
            total_dist_final
            valid_final
            quantum_ran
            all_meta_qaoa
        """
        wall_start = time.perf_counter()

        # Stage A
        candidates, best_seed_raw = self.stage_a_seeds()
        dist_seed_raw = best_seed_raw["total_dist"]
        method_seed = best_seed_raw["method"]

        # Stage B
        candidates_clean, best_seed_clean = self.stage_b_cleanup(candidates)
        dist_after_cleanup = best_seed_clean["total_dist"]
        cleanup_gain = round(dist_seed_raw - dist_after_cleanup, 6)

        # Stage C
        routes_repaired, repair_gain = self.stage_c_repair(best_seed_clean)
        dist_after_repair = total_solution_distance(routes_repaired, self.D)

        # Stage D
        quantum_ran = False
        qaoa_meta = []
        qaoa_gain = 0.0
        routes_final = routes_repaired

        if run_quantum:
            quantum_ran = True
            routes_final, qaoa_gain, qaoa_meta = self.stage_d_qaoa(routes_repaired)

        dist_final = total_solution_distance(routes_final, self.D)
        valid_final = validate_solution(routes_final, self.demands, self.capacity,
                                        self.num_customers, self.num_vehicles)

        wall_time = time.perf_counter() - wall_start
        self._log(f"Done: dist={dist_final:.4f}, valid={valid_final}, "
                  f"total_time={wall_time:.2f}s")

        return {
            "instance_id": self.instance_id,
            "stages": {
                "seed_best_method": method_seed,
                "dist_after_seed": dist_seed_raw,
                "dist_after_cleanup": dist_after_cleanup,
                "cleanup_gain": cleanup_gain,
                "dist_after_repair": round(dist_after_repair, 6),
                "repair_gain": repair_gain,
                "dist_after_qaoa": round(dist_final, 6),
                "qaoa_gain": qaoa_gain,
            },
            "routes_final": routes_final,
            "total_dist_final": round(dist_final, 6),
            "valid_final": valid_final,
            "vehicles_used": len(routes_final),
            "vehicles_allowed": self.num_vehicles,
            "quantum_ran": quantum_ran,
            "all_meta_qaoa": qaoa_meta,
            "wall_time_s": round(wall_time, 3),
        }


# ---------------------------------------------------------------------------
# Convenience: run on a list of instances and collect metrics
# ---------------------------------------------------------------------------

def run_hybrid_batch(instances, run_quantum=True, **solver_kwargs):
    """
    Run HybridSolver on a list of parsed instances.

    Parameters
    ----------
    instances    : list[dict]   -- from common.load_instances_json
    run_quantum  : bool
    **solver_kwargs : passed to HybridSolver constructor

    Returns
    -------
    list[dict]   -- one result dict per instance
    """
    results = []
    for inst in instances:
        solver = HybridSolver(inst, **solver_kwargs)
        r = solver.solve(run_quantum=run_quantum)
        results.append(r)
    return results



## Step 7: Pipeline Evaluation & Execution
Here we run the solver on all 4 challenge instances, showcasing the strict QAOA optimization loop.


In [ ]:

# Instantiate and solve
instances = challenge_instances()
results = {}

for inst in instances:
    print(f"\n{'='*50}")
    print(f"Solving {inst['instance_id']}")
    print(f"{'='*50}")
    # Force QAOA for neighborhoods by setting a suitable budget 
    # (By default it dynamically applies k^2 = budget)
    solver = HybridSolver(inst, max_local_qaoa_qubits=25, verbose=True)
    res = solver.solve(run_quantum=True)
    results[inst['instance_id']] = res
    print(f"\nFINAL VALID: {res['valid_final']} | DIST: {res['total_dist_final']:.4f}")

# Display standard submittable output format
print("\n\n" + "="*50)
print("FINAL CHALLENGE SUBMISSION FORMAT")
print("="*50)
for iid, r in results.items():
    print(format_routes_text(r['routes_final'], instance_id=iid))
    print()



## Step 8: Visualization


In [ ]:

def plot_routes(inst, routes, title=''):
    nodes = inst['nodes']
    depot = nodes[0]
    cust_xy = nodes[1:]
    D_plot = build_distance_matrix(nodes)
    total_d = total_solution_distance(routes, D_plot)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(*depot, c='black', marker='X', s=200, zorder=5, label='Depot')
    ax.text(depot[0]+0.3, depot[1]+0.3, '0', fontsize=10)

    cx = [p[0] for p in cust_xy]
    cy = [p[1] for p in cust_xy]
    ax.scatter(cx, cy, c='steelblue', s=70, zorder=4)
    for idx, (x, y) in enumerate(cust_xy, start=1):
        ax.text(x+0.3, y+0.3, str(idx), fontsize=8)

    colors = plt.cm.tab10.colors
    for ridx, route in enumerate(routes, start=1):
        pts = np.array([depot] + [cust_xy[c-1] for c in route] + [depot])
        ax.plot(pts[:,0], pts[:,1], linewidth=2.2,
                color=colors[ridx % len(colors)], label=f'R{ridx}: {route}')

    ax.set_title(f"{inst['instance_id']} -- {title}\ndist={total_d:.4f}", fontsize=13)
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

for iid, r in results.items():
    inst_plot = next(i for i in instances if i['instance_id'] == iid)
    plot_routes(inst_plot, r['routes_final'], title='Hybrid QAOA Solver')

